# Supply Chain Risk Prediction: Feature Engineering
Transforming cleaned data into numerical features suitable for machine learning.

In [1]:
# --- Setup & Load Cleaned Data ---
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler

df_model = pd.read_csv("../data/cleaned/cleaned_data.csv")

## Date Features
Extracting useful components from the order date.

In [2]:
# --- Date Features ---
df_model["order date (DateOrders)"] = pd.to_datetime(df_model["order date (DateOrders)"], errors="coerce")
df_model["order_dow"]        = df_model["order date (DateOrders)"].dt.dayofweek
df_model["order_month"]      = df_model["order date (DateOrders)"].dt.month
df_model["order_is_weekend"] = df_model["order_dow"].isin([5, 6]).astype(int)

## Chronological Train/Test Split
Splitting data by time to simulate real-world predictive environments.

In [3]:
# --- Chronological Train/Test Split ---
df_model = df_model.sort_values("order date (DateOrders)").reset_index(drop=True)
split_point = int(len(df_model) * 0.8)

train_df = df_model.iloc[:split_point].copy()
test_df  = df_model.iloc[split_point:].copy()

train_df = train_df.drop(columns=["order date (DateOrders)"])
test_df  = test_df.drop(columns=["order date (DateOrders)"])

## Target Encoding & Outlier Handling

In [4]:
# --- Historical Late-Delivery Rate ---
group_cols = ["Customer Segment", "Order Region", "Shipping Mode"]
overall_late_rate = train_df["Late_delivery_risk"].mean()
historical_rate_maps = {}

for group_col in group_cols:
    rate_lookup = train_df.groupby(group_col)["Late_delivery_risk"].mean()
    historical_rate_maps[group_col] = rate_lookup
    train_df[f"{group_col}_late_rate"] = train_df[group_col].map(rate_lookup).fillna(overall_late_rate)
    test_df[f"{group_col}_late_rate"]  = test_df[group_col].map(rate_lookup).fillna(overall_late_rate)

# --- Handle Outliers ---
outlier_cols = ["Sales", "Order Item Quantity", "Order Item Product Price", "Order Item Discount"]
outlier_bounds = {}

for col in outlier_cols:
    lower_bound = train_df[col].quantile(0.01)
    upper_bound = train_df[col].quantile(0.99)
    outlier_bounds[col] = (lower_bound, upper_bound)
    train_df[col] = train_df[col].clip(lower_bound, upper_bound)
    test_df[col]  = test_df[col].clip(lower_bound, upper_bound)

## Feature Interactions & Encoding
Creating combined features and encoding categoricals.

In [5]:
# --- Feature Interactions ---
for dataset in (train_df, test_df):
    dataset["mode_region_interaction"] = dataset["Shipping Mode"].astype(str) + "_" + dataset["Order Region"].astype(str)
    dataset["qty_x_discount_rate"] = dataset["Order Item Quantity"] * dataset["Order Item Discount Rate"]

# --- Categorical Encoding ---
low_cardinality_cols = ["Type", "Customer Segment", "Shipping Mode", "Market", "Order Region", "Category Name"]
low_cardinality_cols = [c for c in low_cardinality_cols if c in train_df.columns]

high_cardinality_cols = ["Customer City", "Customer Country", "Customer State", "Order City", "Order Country", "Order State", "Department Name", "mode_region_interaction", "Order Status"]
high_cardinality_cols = [c for c in high_cardinality_cols if c in train_df.columns]

frequency_maps = {}
for col in high_cardinality_cols:
    freq_lookup = train_df[col].value_counts(normalize=True)
    frequency_maps[col] = freq_lookup
    train_df[col + "_freq"] = train_df[col].map(freq_lookup)
    test_df[col + "_freq"]  = test_df[col].map(freq_lookup).fillna(0)

train_df = train_df.drop(columns=high_cardinality_cols)
test_df  = test_df.drop(columns=high_cardinality_cols)

train_df = pd.get_dummies(train_df, columns=low_cardinality_cols, drop_first=True)
test_df  = pd.get_dummies(test_df, columns=low_cardinality_cols, drop_first=True)
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

## Scaling and Saving Artifacts

In [6]:
# --- Scaling ---
scale_cols = ["Sales", "Order Item Quantity", "Order Item Product Price", "Order Item Discount", "Order Item Discount Rate", "Sales per customer", "Order Item Total"]
scale_cols = [c for c in scale_cols if c in train_df.columns]

scaler = RobustScaler()
train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])
test_df[scale_cols]  = scaler.transform(test_df[scale_cols])

leftover_text_cols = train_df.select_dtypes(include="object").columns.tolist()
train_df = train_df.drop(columns=leftover_text_cols).fillna(0)
test_df  = test_df.drop(columns=[c for c in leftover_text_cols if c in test_df.columns]).fillna(0)

# --- Save Features & Artifacts ---
X_train = train_df.drop(columns=["Late_delivery_risk"])
y_train = train_df["Late_delivery_risk"]
X_test = test_df.drop(columns=["Late_delivery_risk"])
y_test = test_df["Late_delivery_risk"]

joblib.dump(X_train, '../data/train/X_train.pkl')
joblib.dump(y_train, '../data/train/y_train.pkl')
joblib.dump(X_test, '../data/test/X_test.pkl')
joblib.dump(y_test, '../data/test/y_test.pkl')

joblib.dump(scaler, '../../optichain-backend/model/scaler.pkl')
joblib.dump(list(X_train.columns), '../../optichain-backend/model/model_columns.pkl')
joblib.dump(historical_rate_maps, '../../optichain-backend/model/historical_rate_maps.pkl')
joblib.dump(frequency_maps, '../../optichain-backend/model/frequency_maps.pkl')
joblib.dump(overall_late_rate, '../../optichain-backend/model/overall_late_rate.pkl')
joblib.dump(outlier_bounds, '../../optichain-backend/model/outlier_bounds.pkl')

print("Feature engineering complete. Data and artifacts saved.")

Feature engineering complete. Data and artifacts saved.
